In [1]:
# import os
# import glob
# import gc
# import pandas as pd

# folder_path = r'C:\8ssible-Healing-Seoul-Analysis\branch_SG\8SSIBLE_SG_DATA\일별평균대기오염도 데이터 파일'

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
DATA_DIR = Path(
    r"C:\8ssible-Healing-Seoul-Analysis\branch_SG\8SSIBLE_SG_DATA\일별평균대기오염도 데이터 파일"
)

INPUT_FILES = [
    DATA_DIR / "일별평균대기오염도_2022.csv",
    DATA_DIR / "일별평균대기오염도_2023.csv",
    DATA_DIR / "일별평균대기오염도_2024.csv",
]

OUTPUT_FILE = Path.cwd() / "일별평균대기오염도_2022_2024_통합.csv"
MONTHLY_OUTPUT_FILE = Path.cwd() / "월별평균대기오염도_2022_2024_통합.csv"

In [4]:
pollution_columns = [
    "이산화질소농도(ppm)",
    "오존농도(ppm)",
    "일산화탄소농도(ppm)",
    "아황산가스농도(ppm)",
    "미세먼지농도(㎍/㎥)",
    "초미세먼지농도(㎍/㎥)",
]

In [5]:
def read_csv(path):
    for encoding in ("utf-8-sig", "cp949", "euc-kr"):
        try:
            df = pd.read_csv(path, encoding=encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError(f"파일을 읽을 수 없습니다: {path}")

    df["측정일시"] = pd.to_datetime(df["측정일시"].astype(str), format="%Y%m%d")
    df["연도"] = df["측정일시"].dt.year
    df["월"] = df["측정일시"].dt.month
    df["원본파일"] = path.name

    for col in pollution_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

In [6]:
dataframes = [read_csv(path) for path in INPUT_FILES]

merged_df = pd.concat(dataframes, ignore_index=True)

merged_df.head()

,측정일시,측정소명,이산화질소농도(ppm),오존농도(ppm),일산화탄소농도(ppm),아황산가스농도(ppm),미세먼지농도(㎍/㎥),초미세먼지농도(㎍/㎥),연도,월,원본파일
0,2022-01-01,강남구,0.029,0.014,0.5,0.003,25.0,12.0,2022,1,일별평균대기오염도_2022.csv
1,2022-01-01,홍릉로,0.038,0.009,0.6,0.004,27.0,16.0,2022,1,일별평균대기오염도_2022.csv
2,2022-01-01,행주,0.034,0.012,0.7,0.004,27.0,16.0,2022,1,일별평균대기오염도_2022.csv
3,2022-01-01,항동,0.030,0.011,0.5,0.003,27.0,14.0,2022,1,일별평균대기오염도_2022.csv
4,2022-01-01,한강대로,0.037,0.011,0.6,0.003,34.0,14.0,2022,1,일별평균대기오염도_2022.csv


In [7]:
missing_before = pd.DataFrame({
    "결측치 개수": merged_df.isna().sum(),
    "결측치 비율(%)": (merged_df.isna().mean() * 100).round(2)
})

missing_before

,결측치 개수,결측치 비율(%)
측정일시,0,0.00
측정소명,0,0.00
이산화질소농도(ppm),226,0.41
오존농도(ppm),202,0.37
일산화탄소농도(ppm),769,1.40
아황산가스농도(ppm),336,0.61
미세먼지농도(㎍/㎥),274,0.50
초미세먼지농도(㎍/㎥),263,0.48
연도,0,0.00
월,0,0.00


In [8]:
missing_by_station = (
    merged_df
    .groupby("측정소명")[pollution_columns]
    .apply(lambda x: x.isna().sum().sum())
    .sort_values(ascending=False)
)

missing_by_station.head(10)

측정소명
중구      266
강동구     164
동대문구    150
남산      129
도산대로    104
관악산     100
한강대로     90
항동       85
서대문구     81
북한산      80
dtype: int64

In [9]:
filled_df = merged_df.sort_values(["측정소명", "측정일시"]).copy()

filled_df[pollution_columns] = (
    filled_df
    .groupby("측정소명")[pollution_columns]
    .transform(lambda x: x.interpolate(method="linear"))
)

filled_df[pollution_columns] = (
    filled_df
    .groupby("측정소명")[pollution_columns]
    .transform(lambda x: x.ffill().bfill())
)

filled_df.head()

,측정일시,측정소명,이산화질소농도(ppm),오존농도(ppm),일산화탄소농도(ppm),아황산가스농도(ppm),미세먼지농도(㎍/㎥),초미세먼지농도(㎍/㎥),연도,월,원본파일
0,2022-01-01,강남구,0.029,0.014,0.5,0.003,25.0,12.0,2022,1,일별평균대기오염도_2022.csv
99,2022-01-02,강남구,0.027,0.016,0.5,0.003,35.0,22.0,2022,1,일별평균대기오염도_2022.csv
149,2022-01-03,강남구,0.034,0.009,0.5,0.003,25.0,14.0,2022,1,일별평균대기오염도_2022.csv
176,2022-01-04,강남구,0.025,0.016,0.5,0.004,34.0,18.0,2022,1,일별평균대기오염도_2022.csv
200,2022-01-05,강남구,0.037,0.006,0.7,0.004,46.0,27.0,2022,1,일별평균대기오염도_2022.csv


In [10]:
missing_after = pd.DataFrame({
    "결측치 개수": filled_df.isna().sum(),
    "결측치 비율(%)": (filled_df.isna().mean() * 100).round(2)
})

missing_after

,결측치 개수,결측치 비율(%)
측정일시,0,0.0
측정소명,0,0.0
이산화질소농도(ppm),0,0.0
오존농도(ppm),0,0.0
일산화탄소농도(ppm),0,0.0
아황산가스농도(ppm),0,0.0
미세먼지농도(㎍/㎥),0,0.0
초미세먼지농도(㎍/㎥),0,0.0
연도,0,0.0
월,0,0.0


In [11]:
monthly_df = (
    filled_df
    .groupby(["연도", "월"], as_index=False)[pollution_columns]
    .mean()
    .round(4)
)

monthly_df

,연도,월,이산화질소농도(ppm),오존농도(ppm),일산화탄소농도(ppm),아황산가스농도(ppm),미세먼지농도(㎍/㎥),초미세먼지농도(㎍/㎥)
0,2022,1,0.0330,0.0158,0.6834,0.0038,43.8009,28.3790
1,2022,2,0.0274,0.0260,0.5609,0.0033,40.2862,25.3082
2,2022,3,0.0278,0.0271,0.5105,0.0031,42.6626,21.6339
3,2022,4,0.0222,0.0409,0.4357,0.0030,46.5050,21.8740
4,2022,5,0.0185,0.0467,0.3949,0.0029,35.0671,16.9890
5,2022,6,0.0159,0.0343,0.3640,0.0024,23.6500,12.2440
6,2022,7,0.0169,0.0355,0.3816,0.0025,25.6335,15.6023
7,2022,8,0.0161,0.0290,0.3685,0.0026,20.7806,10.8845
8,2022,9,0.0181,0.0298,0.3827,0.0026,22.6391,11.0663
9,2022,10,0.0218,0.0223,0.4561,0.0028,25.7895,13.6371


In [12]:
filled_df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
monthly_df.to_csv(MONTHLY_OUTPUT_FILE, index=False, encoding="utf-8-sig")

print("결측치 처리된 통합 파일:", OUTPUT_FILE)
print("월별 평균 파일:", MONTHLY_OUTPUT_FILE)

결측치 처리된 통합 파일: c:\8ssible-Healing-Seoul-Analysis\branch_SG\일별평균대기오염도_2022_2024_통합.csv
월별 평균 파일: c:\8ssible-Healing-Seoul-Analysis\branch_SG\월별평균대기오염도_2022_2024_통합.csv
